# Lab 01: Fitting a Line When Your Data Are Lying to You

**ASTR 457, Fall 2026, posted Thu Sep 3, due Wed Sep 9 by Noon (fork → PR, as always)**

*Why this lab: fitting a line to contaminated data is the statistic behind real discoveries (it's how Gaia BH1's dark companion was found), and refereeing someone else's fit, including a machine's, is the skill you'll use in every collaboration, every review, and every AI-assisted analysis for the rest of your career.*

Every one of you has your own dataset: `data/<your netid>.csv`, with columns `x`, `y`, `sigma_y`.
It was generated from a straight line, $y = mx + b$, with Gaussian noise of the stated
$\sigma_y$, except that some fraction of the points are **outliers** drawn from a much
broader distribution. I know the true $m$, $b$, and outlier fraction for your dataset.
You don't, and you can't look them up, and your classmates' values are different from yours.

**How this is graded.** Not on whether your code runs, but on whether your answers are *right*
and your uncertainties are *honest*. Part of your grade comes from how close your reported
$m$ and $b$ are to your truth **in units of your own reported uncertainty**. Report tiny
error bars you can't back up and you will lose points even if your central value is close.
Report huge error bars to be safe and you'll lose points too. Calibration is the skill.

**AI policy reminder.** Use whatever tools you like, including AI assistants, and document
it in Part 4. You may be selected to defend this lab in person. Welcome to doing research.

## Part 1: The naive fit (20%)

Load your dataset and fit a straight line by **weighted least squares**, using the reported
`sigma_y` and all the data points. Report $m \pm \sigma_m$ and $b \pm \sigma_b$, plot the
data (with error bars) and your fit, and compute the reduced $\chi^2$.

Then answer, in a few sentences: what is the reduced $\chi^2$ telling you, and why is it
doing that? Would you trust these error bars? (This fit is *supposed* to be bad. Understanding
exactly how it is bad is the point.)

In [ ]:
# Part 1: your work here
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import ascii
tabledat = ascii.read('/Users/mtscl/ast457_2026_Fall/labs/01/data/ms155.csv', header_start=0, data_start=1, delimiter=',')
x = tabledat["x"]
y = tabledat["y"]
sigma_y = tabledat["sigma_y"]
fitcoeff,fitcov = np.polyfit(x,y,1,w=1/sigma_y,cov=True)
m = fitcoeff[0]
b = fitcoeff[1]
sigma_m = np.sqrt(fitcov[0][0])
sigma_b = np.sqrt(fitcov[1][1])
fitys = np.polyval(fitcoeff,x)
fitresid = y - fitys
chi2 = np.sum((fitresid/sigma_y)**2)
redchi2 = chi2/(len(x)-2)
plt.errorbar(x,y,sigma_y,fmt="o",label="Data with error in sigma_y")
plt.plot(x,fitys, label="Least Squares Fit")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Least Squares Fit of Data")
plt.legend()
print(f"It seems that with a Least Squares fit, we have m = {m:.2f} +or- {sigma_m:.2f} and b = {b:.2f} +or- {sigma_b:.2f}. We also have a reduced chi squared of {redchi2:.2f}")

**ANSWER (a few sentences):**

*The reduced chi squared is huge, far bigger than 1 making this fit terrible. A reduced chi squared this large suggests that a few points are significantly farther from the fit than their sigma's would suggest. This implies that a few outliers exist and are messing up the fit. I would not trust the error bars as they fail to represent how distant many of the data points are from the fit, as a consequence of a large reduced chi squared.*

## Part 2: A fit you can defend (40%)

Now deal with the outliers. Do this **two ways**:

1. **Sigma-clipping**: iteratively remove points that are discrepant with the fit, refit,
   and repeat until it converges. Be explicit about your clipping threshold and why you chose it.
2. **A mixture model**: model each point as coming from either the line (with its stated
   $\sigma_y$) or from a broader outlier distribution, and maximize the appropriate likelihood
   (see Hogg, Bovy & Lang 2010, §3, for the canonical treatment; this is the same problem
   with different numbers).

For each method report $m \pm \sigma_m$, $b \pm \sigma_b$, and (for the mixture) your
estimate of the outlier fraction. Then commit to a **final answer**, one set of numbers
you'd put in a paper, and justify the choice. State clearly how you estimated your
uncertainties and why you believe them.

In [ ]:
# Part 2: your work here
import numpy.ma as mask
from scipy.optimize import minimize
from scipy.stats import norm
def sigmaclipfit(x,y,sigma,clip=3,iterations=20):
    cut = np.ones_like(x, dtype=bool)
    for _ in range(iterations):
        fitcoeff,fitcov = np.polyfit(x[cut], y[cut], 1, w=1/sigma[cut],cov=True)
        fitresid = (y - np.polyval(fitcoeff, x)) / sigma
        cutnew = np.abs(fitresid) < clip
        if np.array_equal(cutnew, cut):
            break
        cut = cutnew
    return fitcoeff, cut, fitcov
clipcoeff,cut,clipcov = sigmaclipfit(x,y,sigma_y)
mclip = clipcoeff[0]
bclip = clipcoeff[1]
sigma_mclip = clipcov[0][0]
sigma_bclip = clipcov[1][1]
clipys = np.polyval(clipcoeff,x)
clipresid = y[cut] - clipys[cut]
clipchi2 = np.sum((clipresid/sigma_y[cut])**2)
clipredchi2 = clipchi2/(len(x[cut])-2)
plot1 = plt.figure()
plt.errorbar(x[cut],y[cut],yerr=sigma_y[cut],fmt="o",label="Data unclipped")
plt.errorbar(x[~cut],y[~cut],yerr=sigma_y[~cut],fmt="o",color="r",label="Data clipped")
plt.plot(x,clipys, label="Fit with sigma clipping")
plt.plot(x,fitys, label="Least Squares Fit with no clipping", c="r",alpha=.2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fit of Data with sigma clipping")
plt.legend()
plot2 = plt.figure()
plt.scatter(x[cut],clipresid/sigma_y[cut],label="unclipped data")
plt.scatter(x[~cut],(y[~cut]-clipys[~cut])/sigma_y[~cut],label="clipped data")
plt.axhspan(-3, 3, alpha=0.1, label="data within 3 sigma of the fit")
plt.xlabel("x")
plt.ylabel("residual/sigma")
plt.title("Residual plot sigma clipping")
plt.legend()



def neg_log_likelihood(theta, x, y, sigma):
    m, b, Pb, Yb, lnVb = theta
    Pb = np.clip(Pb, 1e-6, 1 - 1e-6)
    Vb = np.exp(lnVb)
    model = m * x + b
    good = (1 - Pb) * norm.pdf(y, loc=model, scale=sigma)
    bad_pop = Pb * norm.pdf(y, loc=Yb, scale=np.sqrt(Vb + sigma**2))
    return -np.sum(np.log(good + bad_pop + 1e-300))
theta0 = [mclip, bclip, 0.2, np.mean(y), np.log(50.0)]
result = minimize(neg_log_likelihood, theta0, args=(x, y, sigma_y), method='Nelder-Mead')
mmix, bmix, Pb_mix, Yb_mix, lnVb_mix = result.x
mixys = np.polyval(np.array([mmix,bmix]),x)
mixresid = y - mixys
mixchi2 = np.sum((mixresid[cut]/sigma_y[cut])**2)
mixredchi2 = mixchi2/(len(x[cut])-2)
plot3 = plt.figure()
plt.errorbar(x[cut],y[cut],yerr=sigma_y[cut],fmt="o",label="previously unclipped data")
plt.errorbar(x[~cut],y[~cut],yerr=sigma_y[~cut],c="r",fmt="o",label="previously clipped data")
plt.plot(x,mixys, label="Fit with Mixture model")
plt.plot(x,fitys, label="Least Squares Fit with Mixture model/clipping", c="r",alpha=.2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fit of Data with Mixture model")
plt.legend()
plot2 = plt.figure()
plt.scatter(x[cut],mixresid[cut]/sigma_y[cut],label="unclipped data")
plt.scatter(x[~cut],(y[~cut]-mixys[~cut])/sigma_y[~cut],label="clipped data")
plt.axhspan(-3, 3, alpha=0.1, label="data within 3 sigma of the fit")
plt.xlabel("x")
plt.ylabel("residual/sigma")
plt.title("Residual plot mixture model")
plt.legend()
print(f"It seems that with Sigma clipping, we have m = {mclip:.2f} +or- {sigma_mclip:.2f} and b = {bclip:.2f} +or- {sigma_bclip:.2f}. We also have a reduced chi squared of {clipredchi2:.2f} and outlier \nfraction {len(x[~cut])/len(x):.2f}. As for the Mixture model, we have m = {mmix:.2f} and b = {bmix:.2f} as well as a reduced chi squared of {mixredchi2:.2f} and outlier fraction of {Pb_mix:.2f}")

**FINAL ANSWER:** $m = $ 2.3 $\pm$ ___ , $b = $ 33.3 $\pm$ ___ , outlier fraction $\approx$ .31

**Justification and uncertainty method (a short paragraph):**

*The results from the sigma clip method seem to be slightly worse than the Mixture model. The reduced chi squared is about the same between both and the slope and intercept lining up better visually with the majority of the plot, including flat trend in the residual plot. One specific data point in the sigma clip bordering within 3 sigma seemed to be decreasing the overall slope, of which seems to be accounted for in the Mixture model. As for uncertainty, I tried clipping points at different values surrounding 3 sigma. Since each there are only so many bordering data points, I found reduced chi squared for surrounding values and that value was closest to 1 for 3 sigma as threshold, reducing my uncertainty as much as possible by finding all surrounding values without cutting too much out*

## Part 3: The referee report (30%)

The file `ai_solution.ipynb` in this directory is a complete, tidy, confident analysis of
this exact problem, produced by an AI assistant. It runs top to bottom without errors, the
plots look professional, and the prose is fluent. It is also wrong, in at least **three**
distinct, substantive ways (cosmetic nitpicks don't count).

Write a referee report:

1. **Identify** at least three substantive statistical errors.
2. **Demonstrate** each one quantitatively on *your* dataset. Show, with a number or a plot,
   what the error does to the inference. "This is bad practice" is not a demonstration.
3. **State** what the correct treatment is (you already built it in Part 2).

One of the best things you can learn this semester is that fluent, error-free-looking
analysis and correct analysis are different things. Referee accordingly.

**REFEREE REPORT:**

*Flaw 1: It failed to Iterate multiple times when clipping data.*

*Flaw 2: It did not consider changing the threshold value to potentially get a better fit.*

*Flaw 3: It simply resized error bars to ensure the reduced chi squared would be exactly 1.*

In [ ]:
# Part 3: demonstrations here
dem1coeff,d1cut,dem1cov = sigmaclipfit(x,y,sigma_y,4,2)
mdem1 = dem1coeff[0]
bdem1 = dem1coeff[1]
sigma_mdem1 = dem1cov[0][0]
sigma_bdem1 = dem1cov[1][1]
dem1ys = np.polyval(dem1coeff,x)
dem1resid = y[d1cut] - dem1ys[d1cut]
dem1chi2 = np.sum((dem1resid/sigma_y[d1cut])**2)
dem1redchi2 = dem1chi2/(len(x[d1cut])-2)
plot1 = plt.figure()
plt.errorbar(x[d1cut],y[d1cut],yerr=sigma_y[d1cut],fmt="o",label="Data unclipped")
plt.errorbar(x[~d1cut],y[~d1cut],yerr=sigma_y[~d1cut],fmt="o",color="r",label="Data clipped")
plt.plot(x,dem1ys, label="Fit with sigma clipping")
plt.plot(x,fitys, label="Least Squares Fit with no clipping", c="r",alpha=.2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Fit of Data with sigma clipping Demo 1")
plt.legend()
plot2 = plt.figure()
plt.scatter(x[d1cut],dem1resid/sigma_y[d1cut],label="unclipped data")
plt.scatter(x[~d1cut],(y[~d1cut]-dem1ys[~d1cut])/sigma_y[~d1cut],label="clipped data")
plt.axhspan(-3, 3, alpha=0.1, label="data within 3 sigma of the fit")
plt.xlabel("x")
plt.ylabel("residual/sigma")
plt.title("Residual plot sigma clipping Demo 1")
plt.legend()



dem2redchi2s = []
for i in [3,4,5,6]:
    dem2coeff,d2cut,dem2cov = sigmaclipfit(x,y,sigma_y,i,2)
    mdem2 = dem2coeff[0]
    bdem2 = dem2coeff[1]
    sigma_mdem2 = dem2cov[0][0]
    sigma_bdem2 = dem2cov[1][1]
    dem2ys = np.polyval(dem2coeff,x)
    dem2resid = y[d2cut] - dem2ys[d2cut]
    dem2chi2 = np.sum((dem2resid/sigma_y[d2cut])**2)
    dem2redchi2 = dem2chi2/(len(x[d2cut])-2)
    dem2redchi2s.append(dem2redchi2)



dem3coeff,dem3cov = np.polyfit(x,y,1,w=1/sigma_y,cov=True)
dem3m = dem3coeff[0]
dem3b = dem3coeff[1]
dem3ys = np.polyval(dem3coeff,x)
dem3resid = y - dem3ys
dem3chi2 = np.sum((dem3resid/(4.01*sigma_y))**2)
dem3redchi2 = dem3chi2/(len(x)-2)
print(f"Reduced chi squared in the 1 iteration flaw 1 is {dem1redchi2:.2f} at 4 threshold compared to 1.84 at 10 iterations. The less iterations, the less the fit and \nreduced chi squared converge away from least squares. Failing to try different threshold values in flaw 2 makes for wildly different reduced chi \nsquared values such as {dem2redchi2s} from 4 different threshold values. Using the least squares fit from part 1, I managed to get a reduced chi squared of {dem3redchi2:.2f} by using flaw 3's method of scale sigma to get 1 on an imperfect fit. \n\n\n\n\n\nThe way to deal with flaw 1 is to iterate sigma clipping multiple times to be safe with clipping and converge on a fit. Flaw 2 is fixed by using multiple values of the threshold and comparing their fits and reduced chi squared in order to find the best threshold to use. Fixing flaw 3 means using other methods to get reduced chi squared closer to 1 without changing inherent values or using methods that would work for any dataset.")

## Part 4: AI-use appendix & verification plan (10%)

1. **AI use**: which tools did you use (Copilot, Claude, ChatGPT, none, ...), for what,
   what did they get wrong, and how did you catch it? Honesty is graded; "I didn't use any"
   is fine if true.
2. **Verification plan**: list the checks you ran before deciding to trust your Part 2
   numbers (e.g., residual plots, posterior/parameter sanity checks, refitting on synthetic
   data you generated with known answers). For each check: what would failure have looked like?

**AI-USE APPENDIX:**

*I did not use AI directly, but I did use the ai_solution notebook to help me understand what sigma_m, and sigma_b meant. Other than that I just used lecture notes and google when I wasen't sure. I promise I will use AI more and more when I feel comfortable with it.*

**VERIFICATION PLAN:**

*I checked both residual plots and parameter checks for reduced chi squared. Not doing residual plots may make it hard to see the general trend of the data and result with a wrong slope. As for not checking parameters, not only would it restrict you from finding a better fit but also could prevent you from noticing mistakes that would mess up the entire parameter and fit as a whole.*